![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 31 -- Lab 2: Sports Photos with Few Labels

An Olympiad-style task with almost no labels: 100 kinds of sport, and you may use only ten labelled photos per class. You cache CLIP embeddings once for the labelled photos, the validation split and the test split, then compare three classifiers that all live on those cached vectors: zero-shot from the class names, kNN on the labelled embeddings, and a linear probe (`nn.Module`, the loop written out) trained in seconds. You read the validation table, score the test split honestly with a confusion matrix, look at the mistakes, and write the submission.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** Kaggle `gpiosenka/sports-classification` (about 445 MB), downloaded by `kagglehub` in the setup cell: `train/`, `valid/` and `test/` folders with one subfolder per sport, 224 x 224 JPEGs; 13,493 training photos (you use 10 per class), 500 validation and 500 test photos (5 per class). CLIP downloads about 600 MB of weights. A GPU embeds the 2,000 photos in under a minute; a CPU takes a few minutes.

---
# Setup

In [ ]:
# kagglehub fetches the Kaggle dataset; Colab has torch, transformers and scikit-learn.
!pip install -q kagglehub

In [ ]:
import os, random, time
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import confusion_matrix
from transformers import CLIPModel, CLIPProcessor
random.seed(0)
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

In [ ]:
path = kagglehub.dataset_download("gpiosenka/sports-classification")
TRAIN_DIR, VAL_DIR, TEST_DIR = [os.path.join(path, s) for s in ("train", "valid", "test")]
CLASSES = sorted(os.listdir(TRAIN_DIR))
print(len(CLASSES), "classes:", CLASSES[:6], "...")
print("training photos:", sum(len(os.listdir(os.path.join(TRAIN_DIR, c))) for c in CLASSES))

---
# Part 1 -- Ten labelled photos per class

The *Which Tool When* slide: tens of labels per class is kNN and linear-probe territory, zero-shot needs none. Build the three lists first.

## Task 1: The three splits

Write `list_split(folder, per_class=None)` that loops over `CLASSES` in order, takes `sorted(os.listdir(...))` of each class folder (the first `per_class` files when given), and returns two lists: the full paths and the class indices. Build `lab_paths, lab_y = list_split(TRAIN_DIR, 10)`, `val_paths, val_y = list_split(VAL_DIR)`, `test_paths, test_y = list_split(TEST_DIR)`, turn the label lists into tensors, and print the three sizes. Show eight random labelled photos with their class names.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(lab_paths) == 1000 and len(val_paths) == 500 and len(test_paths) == 500 and int(lab_y.max()) == 99, 'ten labelled photos for each of 100 classes; 500 validation and 500 test photos'
print('Task 1 passed')

---
# Part 2 -- Cache the embeddings once

The *In Code: Embed Once* slide: the encoder runs once per photo; everything after is table arithmetic. Keep the three tensors.

## Task 2: CLIP embeddings for the three splits

Load `clip_model` and `processor` for `"openai/clip-vit-base-patch32"` as in Lab 1. Write `embed_images(paths, batch_size=32)`: open each batch of files as RGB, run `processor(images=..., return_tensors="pt").to(device)` and `clip_model.get_image_features`, unwrap with `getattr(out, "pooler_output", out)`, collect on the CPU and return `F.normalize(torch.cat(out), dim=1)`. Compute `E_lab`, `E_val`, `E_test` (time it) and print the three shapes.

Syntax hint (the shape of the call, not the answer):

```python
batch = [Image.open(p).convert("RGB") for p in paths[i:i + batch_size]]
inputs = processor(images=batch, return_tensors="pt").to(device)
feats = clip_model.get_image_features(**inputs)
feats = getattr(feats, "pooler_output", feats)
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tuple(E_lab.shape) == (1000, 512) and tuple(E_val.shape) == (500, 512) and tuple(E_test.shape) == (500, 512), 'three tables of unit-length CLIP embeddings: (1000, 512), (500, 512), (500, 512)'
print('Task 2 passed')

In [ ]:
torch.save({"E_lab": E_lab, "E_val": E_val, "E_test": E_test}, "sports_clip_embeddings.pt")   # the encoder never runs again
print("saved")

---
# Part 3 -- Three classifiers on the same vectors

The *Zero-Shot*, *kNN on Embeddings* and *Linear Probe on Cached Embeddings* slides. Every method below is scored on the same validation embeddings.

## Task 3: Zero-shot from the class names

Write `text_embeddings(sentences)` that runs `clip_model.get_text_features(**processor(text=sentences, return_tensors="pt", padding=True).to(device))`, unwraps, moves to the CPU and normalises. Build `T = text_embeddings([f"a photo of {c}" for c in CLASSES])`, predict `val_pred_zs = (E_val @ T.T).argmax(dim=1)` and print `acc_zs`, the validation accuracy. Then try the bare class names as sentences and print that accuracy too. (The softmax does not change the argmax, so it can be left out when only the class is needed.)

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tuple(T.shape) == (100, 512) and acc_zs > 0.6, 'zero-shot with 100 prompts should pass 0.6 on the validation split'
print('Task 3 passed')

## Task 4: kNN on the labelled embeddings

Write `knn_predict(E_query, E_ref, y_ref, k)`: `S = E_query @ E_ref.T`, the `k` most similar reference rows per query (`topk(k, dim=1).indices`), their labels `y_ref[idx]`, and the majority with `torch.mode(dim=1).values`. Print the validation accuracy for `k = 1` and `k = 5` and keep the better one as `acc_knn`.

Syntax hint (the shape of the call, not the answer):

```python
S = E_query @ E_ref.T
idx = S.topk(k, dim=1).indices
votes = y_ref[idx]                       # (N, k)
pred = torch.mode(votes, dim=1).values
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert acc_knn > 0.6, 'kNN on 1,000 labelled CLIP embeddings should pass 0.6 on validation'
print('Task 4 passed')

## Task 5: The linear probe, trained in seconds

Define `class Probe(nn.Module)` with `self.fc = nn.Linear(512, len(CLASSES))` and a `forward` that returns `self.fc(e)`. Build `loader = DataLoader(TensorDataset(E_lab, lab_y), batch_size=64, shuffle=True)`, `criterion = nn.CrossEntropyLoss()` and `optimizer = torch.optim.Adam(probe.parameters(), lr=1e-2)`. Train for 60 epochs with the loop written out (the four moves on each batch of embeddings); after every epoch compute the validation accuracy from `probe(E_val).argmax(dim=1)` under `torch.no_grad()`, and keep the best accuracy as `acc_probe` and the best weights with `copy.deepcopy(probe.state_dict())`. Print the accuracy every 10 epochs and the time taken.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert acc_probe > 0.7 and sum(p.numel() for p in probe.parameters()) == 51300, 'the probe has 512 * 100 + 100 = 51,300 parameters and should pass 0.7 on validation'
print('Task 5 passed')

## Task 6: The comparison table

Build a DataFrame `table` with the rows `zero-shot (no labels)`, `kNN (10 per class)`, `linear probe (10 per class)` and their validation accuracies, print it, and draw it as a bar chart with the accuracy written above each bar. Which method wins with ten labels per class, and by how much over zero-shot?

In [ ]:
# Your code here


---
# Part 4 -- The honest score, the mistakes, the submission

The test split was never used: score the best method on it, look, write the file.

## Task 7: Test accuracy and the most confused sports

Predict the test split with the probe (`probe(E_test).argmax(dim=1)` under `torch.no_grad()`) as `test_pred`, print the test accuracy, then compute `cm = confusion_matrix(test_y, test_pred, labels=range(100))`, set its diagonal to zero (`np.fill_diagonal(cm, 0)`), and print the five largest off-diagonal entries as `true -> predicted: count` (`np.argsort(cm, axis=None)[::-1][:5]` with `np.unravel_index`). A 100 x 100 matrix is too big to draw; the five worst cells say what it would show.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert cm.shape == (100, 100) and test_acc > 0.6, 'a 100 x 100 confusion matrix; test accuracy above 0.6'
print('Task 7 passed')

## Task 8: Look at the mistakes

Show the first eight test photos where `test_pred != test_y`, titled `true ..., predicted ...` (`fontsize=9`). Are the confused pairs visually close sports?

In [ ]:
# Your code here


## Task 9: Write the submission

Build a DataFrame with `image`, the path of each test photo relative to the test folder (`os.path.relpath(p, TEST_DIR)`), and `label`, the predicted sport name; save it as `submission.csv` without the index, read it back and print its shape and first five rows.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
check = pd.read_csv('submission.csv')
assert list(check.columns) == ['image', 'label'] and len(check) == 500 and set(check['label']) <= set(CLASSES), 'submission.csv: 500 rows, columns image and label, labels among the 100 sport names'
print('Task 9 passed')

---
## Task 10: Reflect

Task 6 put three accuracies side by side: which won, and does the order agree with the *Evidence* slide (zero-shot CLIP equals a 4-shot probe on average, loses to 16-shot)? The probe trained 51,300 parameters on 1,000 vectors in seconds: why is that so much faster than Day 30's probe on the same photos, and what did you give up (name two things)? Task 3 compared two prompts: write a third prompt you would expect to do better for sports, and say why. If the organisers added ten new sports tomorrow with ten photos each, which of the three methods needs no retraining at all, and what would you do for the other two? Finally, which step of this notebook would you cache to disk before a 6-hour competition, and why?

*Your answers here*

---
## Conclusion

- **Few labels, cached embeddings**: ten labelled photos per class for 100 sports, CLIP embeddings computed once for 2,000 photos and saved.
- **Three classifiers on one table**: zero-shot from prompts (no labels), kNN by majority of neighbours (no training), a linear probe as an `nn.Module` trained with the written-out loop in seconds; the three compared on the same validation split.
- **Score, look, submit**: the test split scored with the best method, the most confused sports read from a 100 x 100 confusion matrix, the mistakes shown, and a submission with one row per photo.

---

Made By **Sattam Altwaim**